# Burundi VBR — Data Quality Analysis

Systematic quality checks on `quantity_data_ext_1505.csv`: declared / verified / validated values per org unit, period, and service.

## 0 — Setup & Load

In [1]:
import polars as pl

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(60)

DATA_PATH = "quantity_data_ext_1505.csv"

df_full_all_cols = pl.read_csv(DATA_PATH, infer_schema_length=10000)


In [2]:
RELEVANT_COLS = [
    "dec",
    "ver",
    "val",
    "dec_fbp",
    "ver_fbp",
    "val_fbp",
    "dec_mfp",
    "ver_mfp",
    "val_mfp",
    "dec_cam",
    "ver_cam",
    "val_cam",
    "tarif",
    "ou",
    "month",
    "service",
]

KEY_COLS = [
    "dec",
    "ver",
    "val",
    "tarif",
]

QUANTITY_COLS = [
    "dec",
    "ver",
    "val",
    "dec_fbp",
    "ver_fbp",
    "val_fbp",
    "dec_mfp",
    "ver_mfp",
    "val_mfp",
    "dec_cam",
    "ver_cam",
    "val_cam",
]

problematic_services = [
    "Nouvelle Consultations Curative Femme Enceinte",
    "Prise en charge du nouveau né d'une femme VIH+",
]

problematic_ous = [
    "Z8VIBQuCYbP",
    "lofaiDaHnjx",
    "tTkVCY0EpWe",
    "GhbjTEaxBHR",
    "iB1ilF3YWKR",
    "raCfTlW9EFj",
    "BNk2xoulgMT",
    "yZwZEul3BGu",
    "wcl6ZTUBdW6",
    "B1iTebT0vrW",
    "ay3MRQIuoEP",
    "YVDncsgsS5r",
    "voGl3Kchuk2",
    "S9UaT4UtQ2m",
]


def present(col):
    return pl.col(col).is_not_null() & (pl.col(col) != 0)


def absent(col):
    return pl.col(col).is_null() | (pl.col(col) == 0)


df_full = df_full_all_cols.select(RELEVANT_COLS)


In [3]:
display(f"Shape full: {df_full_all_cols.shape[0]:,} rows × {df_full_all_cols.shape[1]} columns")
display(f"Columns full: {df_full_all_cols.columns}")

'Shape full: 168,140 rows × 43 columns'

"Columns full: ['ou', 'month', 'service', 'ver', 'val', 'dec_cam', 'dec', 'dec_mfp', 'tarif', 'type_contrat', 'type_ou', 'contract_start_date', 'contract_end_date', 'level_1_name', 'level_2_name', 'level_3_name', 'level_4_name', 'level_5_name', 'level_6_name', 'level_1_uid', 'level_2_uid', 'level_3_uid', 'level_4_uid', 'level_5_uid', 'level_6_uid', 'level', 'dhis2_is_not_verified', 'dec_fbp', 'ver_fbp', 'ver_cam', 'ver_mfp', 'val_fbp', 'val_cam', 'val_mfp', 'gain_verif', 'subside_sans_verification', 'subside_avec_verification', 'quarter', 'ecart_dec_ver', 'ecart_ver_val', 'taux_validation', 'weighted_ecart_dec_val', 'ecart_dec_val']"

## 1 - Precleaning
- We get rid of the rows where there is no declared/verified/validated data
- We get rid of the months where we have no verified/validated data (202603)
- We get rid of the services that have no declared data ("Nouvelle Consultations Curative Femme Enceinte", "Prise en charge du nouveau né d'une femme VIH+")

In [4]:
rows_no_data = df_full.filter(absent("dec") & absent("ver") & absent("val"))
display(f"Rows with no data: {rows_no_data.shape[0]:,} rows")
display(rows_no_data.head())
df = df_full.filter(present("dec") | present("ver") | present("val"))

'Rows with no data: 75,572 rows'

dec,ver,val,dec_fbp,ver_fbp,val_fbp,dec_mfp,ver_mfp,val_mfp,dec_cam,ver_cam,val_cam,tarif,ou,month,service
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,i64,str
null,null,null,null,null,null,null,null,null,null,null,null,0.0,"""HXtdYDakoc4""",202501,"""Journée d'hospitalisation >= 5 ans"""
null,null,null,null,null,null,null,null,null,null,null,null,null,"""HXtdYDakoc4""",202501,"""Journée d'hospitalisation < 5 ans"""
null,0.0,0.0,null,null,null,null,null,null,null,null,null,0.0,"""HXtdYDakoc4""",202501,"""Référence et patient arrivé à l'hopital"""
null,0.0,0.0,null,null,null,null,null,null,null,null,null,0.0,"""HXtdYDakoc4""",202501,"""Enfants complétement vaccinés (VAR2)"""
null,0.0,0.0,null,null,null,null,null,null,null,null,null,null,"""HXtdYDakoc4""",202501,"""Femmes enceintes VAT completement vacciné"""


In [5]:
for col in ["dec", "ver", "val"]:
    services_no_data = (
        df.group_by("service")
        .agg(pl.col(col).max().alias(f"max_{col}"))
        .filter(pl.col(f"max_{col}").is_null() | (pl.col(f"max_{col}") == 0))
    )
    display(f"Services with no data in column {col}:")
    display(services_no_data)
display(f"Filtering out problematic services: {problematic_services}")
display(f"Dropping {df.filter(pl.col('service').is_in(problematic_services)).shape[0]:,} rows")
df = df.filter(~pl.col("service").is_in(problematic_services))

'Services with no data in column dec:'

service,max_dec
str,f64
"""Nouvelle Consultations Curative Femme Enceinte""",null
"""Prise en charge du nouveau né d'une femme VIH+""",null


'Services with no data in column ver:'

service,max_ver
str,f64


'Services with no data in column val:'

service,max_val
str,f64


'Filtering out problematic services: [\'Nouvelle Consultations Curative Femme Enceinte\', "Prise en charge du nouveau né d\'une femme VIH+"]'

'Dropping 43 rows'

In [6]:
for col in ["dec", "ver", "val"]:
    months_no_data = (
        df.group_by("month")
        .agg(pl.col(col).max().alias(f"max_{col}"))
        .filter(pl.col(f"max_{col}").is_null() | (pl.col(f"max_{col}") == 0))
    )
    display(f"Months with no data in column {col}:")
    display(months_no_data)
display("Filtering out months with no ver/val data: 202603")
display(f"Dropping {df.filter(pl.col('month').is_in([202603])).shape[0]:,} rows")
df = df.filter(~pl.col("month").is_in([202603]))

'Months with no data in column dec:'

month,max_dec
i64,f64


'Months with no data in column ver:'

month,max_ver
i64,f64
202603,null


'Months with no data in column val:'

month,max_val
i64,f64
202603,null


'Filtering out months with no ver/val data: 202603'

'Dropping 5,779 rows'

In [7]:
count_before_any_dropping = df.shape[0]
for col in ["dec", "ver", "val"]:
    ous_no_data = (
        df.group_by("ou")
        .agg(pl.col(col).max().alias(f"max_{col}"))
        .filter(pl.col(f"max_{col}").is_null() | (pl.col(f"max_{col}") == 0))
    )
    display(f"OUs with no data in column {col}:")
    display(ous_no_data)
display(f"Filtering out ous with either no dec data or no ver/val data: {problematic_ous}")
display(f"Dropping {df.filter(pl.col('ou').is_in(problematic_ous)).shape[0]:,} rows")
df = df.filter(~pl.col("ou").is_in(problematic_ous))

'OUs with no data in column dec:'

ou,max_dec
str,f64
"""wcl6ZTUBdW6""",null
"""BNk2xoulgMT""",null
"""GhbjTEaxBHR""",null
"""lofaiDaHnjx""",null
"""raCfTlW9EFj""",null
"""Z8VIBQuCYbP""",null
"""yZwZEul3BGu""",null
"""tTkVCY0EpWe""",null
"""iB1ilF3YWKR""",null


'OUs with no data in column ver:'

ou,max_ver
str,f64
"""B1iTebT0vrW""",null
"""S9UaT4UtQ2m""",null
"""ay3MRQIuoEP""",null
"""voGl3Kchuk2""",null
"""YVDncsgsS5r""",null


'OUs with no data in column val:'

ou,max_val
str,f64
"""voGl3Kchuk2""",null
"""YVDncsgsS5r""",null
"""B1iTebT0vrW""",null
"""ay3MRQIuoEP""",null
"""S9UaT4UtQ2m""",null


"Filtering out ous with either no dec data or no ver/val data: ['Z8VIBQuCYbP', 'lofaiDaHnjx', 'tTkVCY0EpWe', 'GhbjTEaxBHR', 'iB1ilF3YWKR', 'raCfTlW9EFj', 'BNk2xoulgMT', 'yZwZEul3BGu', 'wcl6ZTUBdW6', 'B1iTebT0vrW', 'ay3MRQIuoEP', 'YVDncsgsS5r', 'voGl3Kchuk2', 'S9UaT4UtQ2m']"

'Dropping 841 rows'

In [8]:
display(f"Shape filtered: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(f"Columns filtered: {df.columns}")

'Shape filtered: 85,905 rows × 16 columns'

"Columns filtered: ['dec', 'ver', 'val', 'dec_fbp', 'ver_fbp', 'val_fbp', 'dec_mfp', 'ver_mfp', 'val_mfp', 'dec_cam', 'ver_cam', 'val_cam', 'tarif', 'ou', 'month', 'service']"

## 2 — Overview

In [9]:
display("=== Period range ===")
display(f"  Months : {sorted(df['month'].unique().to_list())}")
display()
display(f"  Unique org units : {df['ou'].n_unique():,}")
display(f"  Unique services  : {df['service'].n_unique():,}")

'=== Period range ==='

'  Months : [202501, 202502, 202503, 202504, 202505, 202506, 202507, 202508, 202509, 202510, 202511, 202512, 202601, 202602]'

'  Unique org units : 729'

'  Unique services  : 13'

## 3 -- No declared or ver/val data
Note that there are some very clear discrepancies on the lack of information for some org units and services (periods seems more or less consistent). We should raise this

In [10]:
n_total = len(df)

# --- Total columns (null and 0 both treated as absent) ---
ver_val_no_dec = df.filter((present("ver") | present("val")) & absent("dec"))
dec_no_ver_val = df.filter(present("dec") & absent("ver") & absent("val"))

display(
    "=== Total (dec / ver / val) — missing counterparts (null and 0 = absent). Dropping them ==="
)
display(
    f"  ver or val present but dec null/0 : {len(ver_val_no_dec):>7,}  ({100 * len(ver_val_no_dec) / n_total:.1f}%)"
)
display(
    f"  dec present but ver & val null/0 : {len(dec_no_ver_val):>7,}  ({100 * len(dec_no_ver_val) / n_total:.1f}%)"
)


'=== Total (dec / ver / val) — missing counterparts (null and 0 = absent). Dropping them ==='

'  ver or val present but dec null/0 :   1,807  (2.1%)'

'  dec present but ver & val null/0 :   3,201  (3.7%)'

In [11]:
monthly_totals = df.group_by("month").agg(pl.len().alias("total"))
service_total = df.group_by("service").agg(pl.len().alias("total"))
ou_totals = df.group_by("ou").agg(pl.len().alias("total"))

In [12]:
display("=== ver/val without dec — breakdown by month ===")

display(
    ver_val_no_dec.group_by("month")
    .agg(pl.len().alias("count"))
    .join(monthly_totals, on="month")
    .with_columns((pl.col("count") / pl.col("total") * 100).round(2).alias("percentage"))
    .sort("percentage", descending=True)
    .head(10)
)

display("\n=== dec without ver/val — breakdown by month ===")
display(
    dec_no_ver_val.group_by("month")
    .agg(pl.len().alias("count"))
    .join(monthly_totals, on="month")
    .with_columns((pl.col("count") / pl.col("total") * 100).round(2).alias("percentage"))
    .sort("percentage", descending=True)
    .head(10)
)

display("=== ver/val without dec — breakdown by service ===")
display(
    ver_val_no_dec.group_by("service")
    .agg(pl.len().alias("count"))
    .join(service_total, on="service")
    .with_columns((pl.col("count") / pl.col("total") * 100).round(2).alias("percentage"))
    .sort("percentage", descending=True)
    .head(10)
)

display("\n=== dec without ver/val — breakdown by service ===")
display(
    dec_no_ver_val.group_by("service")
    .agg(pl.len().alias("count"))
    .join(service_total, on="service")
    .with_columns((pl.col("count") / pl.col("total") * 100).round(2).alias("percentage"))
    .sort("percentage", descending=True)
    .head(10)
)

display("=== ver/val without dec — breakdown by ou ===")
display(
    ver_val_no_dec.group_by("ou")
    .agg(pl.len().alias("count"))
    .join(ou_totals, on="ou")
    .with_columns((pl.col("count") / pl.col("total") * 100).round(2).alias("percentage"))
    .sort("percentage", descending=True)
    .head(10)
)

display("\n=== dec without ver/val — breakdown by ou ===")
display(
    dec_no_ver_val.group_by("ou")
    .agg(pl.len().alias("count"))
    .join(ou_totals, on="ou")
    .with_columns((pl.col("count") / pl.col("total") * 100).round(2).alias("percentage"))
    .sort("percentage", descending=True)
    .head(10)
)


'=== ver/val without dec — breakdown by month ==='

month,count,total,percentage
i64,u32,u32,f64
202509,178,6100,2.92
202502,168,6231,2.7
202510,149,6005,2.48
202501,146,6294,2.32
202503,143,6221,2.3
202506,140,6129,2.28
202508,138,6125,2.25
202511,125,5979,2.09
202512,110,6074,1.81


'\n=== dec without ver/val — breakdown by month ==='

month,count,total,percentage
i64,u32,u32,f64
202505,257,6205,4.14
202507,248,6155,4.03
202503,240,6221,3.86
202502,234,6231,3.76
202504,236,6296,3.75
202508,229,6125,3.74
202510,223,6005,3.71
202512,224,6074,3.69
202602,219,5989,3.66


'=== ver/val without dec — breakdown by service ==='

...

service,count,total,percentage
str,u32,u32,f64
"""Nouveau-né d’une femme VIH+ mis sous prophylaxie ARV""",144,610,23.61
"""Femme allaitante VIH+ ayant terminé le programme PTME""",118,524,22.52
"""Accouchement dystocique au CDS""",108,648,16.67
"""Dépistage et PEC MAS sans complications médicales chez les m…",616,5536,11.13
"""Une nuit d'observation""",241,3735,6.45
"""Consultation prénatale recentrée""",390,9298,4.19
"""Petite chirurgie < 5 ans""",275,8315,3.31
"""Une nuit d'observation d'une femme ayant accouché et femme a…",300,9163,3.27
"""FP: Tot. NA + AA des méthodes contraceptives modernes (pilul…",251,8715,2.88


'=== ver/val without dec — breakdown by ou ==='

ou,count,total,percentage
str,u32,u32,f64
"""gpD27j9qF9f""",36,77,46.75
"""MYrCV6htvjP""",18,57,31.58
"""VH1hN0hDSHi""",24,121,19.83
"""ZnJMRtmK4DB""",22,119,18.49
"""lr9EwOFjk8H""",22,122,18.03
"""eTLKz7p7czD""",4,28,14.29
"""RGn8CbACvEF""",11,88,12.5
"""qJgho7ci1GU""",15,121,12.4
"""yBv0ReaU1yH""",15,124,12.1


'\n=== dec without ver/val — breakdown by ou ==='

ou,count,total,percentage
str,u32,u32,f64
"""W7mlceX1VEa""",106,114,92.98
"""Xnt7XVzeScT""",101,109,92.66
"""WJ4B7avpUQW""",114,128,89.06
"""T0eoKlcLx5R""",114,128,89.06
"""gIJ2h2iQHnS""",112,126,88.89
"""c5CDIQYKTx5""",111,125,88.8
"""vtCcRnesQRr""",104,118,88.14
"""q4OK7JgylbU""",95,108,87.96
"""sXA9Mmq1LA0""",102,116,87.93


In [13]:
df = df.filter(
    ~((present("ver") | present("val")) & absent("dec"))
)  # Drop rows with ver/val but no dec
df = df.filter(
    ~(present("dec") & absent("ver") & absent("val"))
)  # Drop rows with dec but no ver/val


## 4 -- Nulls

In [14]:
null_zero_summary = pl.DataFrame(
    {
        "column": KEY_COLS,
        "null_or_zero_count": [
            (df[c].is_null() | df[c].fill_null(0).eq(0)).sum() for c in KEY_COLS
        ],
        "null_or_zero_pct": [
            round(100 * (df[c].is_null() | df[c].fill_null(0).eq(0)).sum() / len(df), 1)
            for c in KEY_COLS
        ],
        "valid_count": [(df[c].is_not_null() & df[c].fill_null(1).ne(0)).sum() for c in KEY_COLS],
    }
)

display("=== Null or zero counts for key columns ===")
display(null_zero_summary)
display(
    "There are a lot of rows with val = 0, but this is because of the service not being validated. "
    "There is nothing to clean here, just something to show in the dashboard. "
)

'=== Null or zero counts for key columns ==='

column,null_or_zero_count,null_or_zero_pct,valid_count
str,i64,f64,i64
"""dec""",0,0.0,80897
"""ver""",0,0.0,80897
"""val""",8423,10.4,72474
"""tarif""",39,0.0,80858


'There are a lot of rows with val = 0, but this is because of the service not being validated. There is nothing to clean here, just something to show in the dashboard. '

## 5 — Ordering Incoherence (dec ≥ ver ≥ val)

For each category triple, declared should be ≥ verified, which should be ≥ validated.
There are a fair amount of problems with the declared and verified values. It does seem more serious for some org units and services than for others, but we should raise this.

In [15]:
display("=== Ordering violations ===")
ver_lt_val = df.filter(pl.col("ver") < pl.col("val"))
pct_va = 100 * len(ver_lt_val) / len(df) if len(df) > 0 else 0
display(f"  | total  ver < total val : {len(ver_lt_val):>6,} / {len(df):>7,} ({pct_va:5.1f}%)")

'=== Ordering violations ==='

'  | total  ver < total val :      4 /  80,897 (  0.0%)'

In [16]:
dec_lt_ver = df.filter(pl.col("dec") < pl.col("ver"))
dec_lt_ver = dec_lt_ver.with_columns(
    ((pl.col("ver") - pl.col("dec")) / pl.col("dec") * 100).alias("pct_diff")
)
display(
    f"  total dec < total ver : {len(dec_lt_ver):>6,} / {len(df):>7,} ({100 * len(dec_lt_ver) / len(df):5.1f}%) "
    f"  | total  dec < total ver real problem : "
    f"{len(dec_lt_ver.filter(pl.col('pct_diff') > 10)):>6,} / {len(df):>7,} ({100 * len(dec_lt_ver.filter(pl.col('pct_diff') > 10)) / len(df):5.1f}%)"
)


'  total dec < total ver : 11,101 /  80,897 ( 13.7%)   | total  dec < total ver real problem :  3,639 /  80,897 (  4.5%)'

In [17]:
real_order_problems = dec_lt_ver.filter(pl.col("pct_diff") > 10)

display("=== dec < ver (diff > 10%) — breakdown by service ===")
display(
    real_order_problems.group_by("service")
    .agg(pl.len().alias("count"))
    .join(service_total, on="service")
    .with_columns((pl.col("count") / pl.col("total") * 100).round(2).alias("percentage"))
    .sort("percentage", descending=True)
    .head(10)
)

display("\n=== dec < ver (diff > 10%) — breakdown by ou ===")
display(
    real_order_problems.group_by("ou")
    .agg(pl.len().alias("count"))
    .join(ou_totals, on="ou")
    .with_columns((pl.col("count") / pl.col("total") * 100).round(2).alias("percentage"))
    .sort("percentage", descending=True)
    .head(10)
)

display("\n=== dec < ver (diff > 10%) — breakdown by month ===")
display(
    real_order_problems.group_by("month")
    .agg(pl.len().alias("count"))
    .join(monthly_totals, on="month")
    .with_columns((pl.col("count") / pl.col("total") * 100).round(2).alias("percentage"))
    .sort("percentage", descending=True)
    .head(10)
)


'=== dec < ver (diff > 10%) — breakdown by service ==='

service,count,total,percentage
str,u32,u32,f64
"""Petite chirurgie < 5 ans""",757,8315,9.1
"""Consultation prénatale recentrée""",688,9298,7.4
"""Une nuit d'observation""",257,3735,6.88
"""Dépistage et PEC MAS sans complications médicales chez les m…",306,5536,5.53
"""Une nuit d'observation d'une femme ayant accouché et femme a…",489,9163,5.34
"""Nouvelle consultation curative pour les femmes enceintes ou …",518,9944,5.21
"""Nouvelle Consultation Curative (>=5 ans)""",300,10166,2.95
"""FP: Tot. NA + AA des méthodes contraceptives modernes (pilul…",138,8715,1.58
"""Nouvelle Consultation Curative (< 5 ans)""",114,10077,1.13


'\n=== dec < ver (diff > 10%) — breakdown by ou ==='

ou,count,total,percentage
str,u32,u32,f64
"""XSSbJoRDElK""",30,116,25.86
"""rK4yuXZL7d9""",20,112,17.86
"""bj0uFH7rSj4""",22,130,16.92
"""bdu7msZuCIJ""",19,114,16.67
"""VL6UuxVzDQy""",20,123,16.26
"""igaVlKHn3S4""",17,105,16.19
"""RtrhtKWnTKc""",18,112,16.07
"""RZFcp6NUvlK""",8,51,15.69
"""dECRq6FwxHk""",12,77,15.58


'\n=== dec < ver (diff > 10%) — breakdown by month ==='

month,count,total,percentage
i64,u32,u32,f64
202501,339,6294,5.39
202502,316,6231,5.07
202504,293,6296,4.65
202506,282,6129,4.6
202509,277,6100,4.54
202503,274,6221,4.4
202505,270,6205,4.35
202507,252,6155,4.09
202511,236,5979,3.95


## 6 -- CAM and MFP columns
We assume that all of the CAM and MFP services are validated and verified. So, when the CAM and MFP services are a big percentage of the total services, we have weird things happening.

I can just get rid of the rows where we have these values, but then the problem is that "Nouvelle Consultation Curative (>=5 ans)" service is practicaly ignored.

In [18]:
df_cam_mfp = df.with_columns((pl.col("dec_cam") + pl.col("dec_mfp")).alias("dec_cam_mfp"))
df_cam_mfp = df_cam_mfp.filter(pl.col("dec_cam_mfp") > 0)
df_cam_mfp = df_cam_mfp.with_columns(
    (100 * pl.col("dec_cam_mfp") / pl.col("dec")).alias("pct_cam_mfp")
)
df_cam_mfp_big = df_cam_mfp.filter(pl.col("pct_cam_mfp") > 10)
display("=== Percentage of CAM+MFP out of total declared (dec) ===")
display(f"Number of rows: {len(df_cam_mfp)}. ")
display(
    f"Number of rows with cam/mfp values: {(df_cam_mfp['dec_cam_mfp'] > 0).sum()}"
    f" ({100 * (df_cam_mfp['dec_cam_mfp'] > 0).sum() / len(df):.1f}%)"
)
display(
    f"Number of rows with a lot of cam/mfp values: {(df_cam_mfp_big['dec_cam_mfp'] > 0).sum()}"
    f" ({100 * (df_cam_mfp_big['dec_cam_mfp'] > 0).sum() / len(df):.1f}%)"
)

'=== Percentage of CAM+MFP out of total declared (dec) ==='

'Number of rows: 7844. '

'Number of rows with cam/mfp values: 7844 (9.7%)'

'Number of rows with a lot of cam/mfp values: 7665 (9.5%)'

In [19]:
display("=== CAM+MFP > 10% of dec — breakdown by service ===")
display(
    df_cam_mfp_big.group_by("service")
    .agg(pl.len().alias("count"))
    .join(service_total, on="service")
    .with_columns((pl.col("count") / pl.col("total") * 100).round(2).alias("percentage"))
    .sort("percentage", descending=True)
    .head(10)
)

display("\n=== CAM+MFP > 10% of dec — breakdown by ou ===")
display(
    df_cam_mfp_big.group_by("ou")
    .agg(pl.len().alias("count"))
    .join(ou_totals, on="ou")
    .with_columns((pl.col("count") / pl.col("total") * 100).round(2).alias("percentage"))
    .sort("percentage", descending=True)
    .head(10)
)

display("\n=== CAM+MFP > 10% of dec — breakdown by month ===")
display(
    df_cam_mfp_big.group_by("month")
    .agg(pl.len().alias("count"))
    .join(monthly_totals, on="month")
    .with_columns((pl.col("count") / pl.col("total") * 100).round(2).alias("percentage"))
    .sort("percentage", descending=True)
    .head(10)
)


'=== CAM+MFP > 10% of dec — breakdown by service ==='

service,count,total,percentage
str,u32,u32,f64
"""Nouvelle Consultation Curative (>=5 ans)""",7508,10166,73.85
"""Nouvelle consultation curative pour les femmes enceintes ou …",97,9944,0.98
"""Nouvelle Consultation Curative (< 5 ans)""",58,10077,0.58
"""Une nuit d'observation""",1,3735,0.03
"""Une nuit d'observation d'une femme ayant accouché et femme a…",1,9163,0.01


'\n=== CAM+MFP > 10% of dec — breakdown by ou ==='

ou,count,total,percentage
str,u32,u32,f64
"""HfHjigQt4mv""",31,113,27.43
"""pTknOwAkkue""",13,54,24.07
"""B30tqheAWYq""",22,93,23.66
"""tSOm71uwOE7""",14,61,22.95
"""olpOkaoG4IY""",20,92,21.74
"""t1fAcxteooK""",14,71,19.72
"""j1E3FyvOG3R""",14,72,19.44
"""KyopxvU7MMB""",23,120,19.17
"""e5XGDIQbMi2""",18,94,19.15


'\n=== CAM+MFP > 10% of dec — breakdown by month ==='

month,count,total,percentage
i64,u32,u32,f64
202602,552,5989,9.22
202511,546,5979,9.13
202601,555,6102,9.1
202506,555,6129,9.06
202512,548,6074,9.02
202510,541,6005,9.01
202505,553,6205,8.91
202503,553,6221,8.89
202504,557,6296,8.85


## 7 — Duplicate Rows

Duplicates on the broader key `(ou, month, service)` may indicate the same service under two contract types; duplicates on the full key are unambiguous data errors.

In [20]:
KEY_BROAD = ["ou", "month", "service"]

exact_dupes = df.filter(df.is_duplicated())
display(f"Exact full-row duplicates: {len(exact_dupes):,}")

broad_dupes = df.filter(df.select(KEY_BROAD).is_duplicated())
display(
    f"Duplicates on (ou, month, service): {len(broad_dupes):,} rows — may include different type_contrat"
)


'Exact full-row duplicates: 875'

'Duplicates on (ou, month, service): 875 rows — may include different type_contrat'

## 8 Final clearning
Get rid of the rows where dec is a lot smaller than the ver
Get rid of the rows where the CAM and MFP services are a big percentage of the total services

In [21]:
final_data = df.filter(
    ((100 * (pl.col("ver") - pl.col("dec")) / pl.col("dec")) <= 10)
    & (
        (100 * (pl.col("dec_cam").fill_null(0) + pl.col("dec_mfp").fill_null(0)) / pl.col("dec"))
        <= 10
    )
)
display(f"Initial count: {count_before_any_dropping} rows. Final count: {len(final_data):,} rows.")

'Initial count: 86746 rows. Final count: 64,230 rows.'